# YMCM Training with Stratified 5-Fold Cross-Validation

This notebook trains YMCM fold-by-fold using the CV feature folders generated by the feature extraction notebook. It reports mean, variance, standard deviation, and 95% confidence intervals.

In [1]:

# ============================================================
# YMCM Training with Stratified 5-Fold Cross-Validation
# Uses feature arrays created by Features_Extraction_YSG_CV_REVIEWER_READY.ipynb
# ============================================================

import os
import gc
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical, Sequence

from sklearn.metrics import (
    
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# GPU configuration
# ============================================================

def configure_gpu():
    """
    Configure TensorFlow GPU memory growth.
    Run this cell before building any model.
    """
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
    os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

    gpus = tf.config.list_physical_devices("GPU")

    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"GPU memory growth enabled for {len(gpus)} GPU(s).")
        except RuntimeError as e:
            print("GPU was already initialized. If needed, restart kernel and run this cell first.")
            print(e)
    else:
        print("No GPU found. Training will use CPU.")

    print("TensorFlow version:", tf.__version__)


# ============================================================
# Constants
# ============================================================

GENRES = ["adeni", "hadrami", "lahji", "sanaai", "tihami"]
NUM_CLASSES = len(GENRES)
INPUT_SHAPE = (224, 224, 3)

CV_DATA_DIR = "/root/autodl-tmp/cv_split_data/"
OUTPUT_DIR = "/root/autodl-tmp/YMCM_CV_results/"

N_SPLITS = 5
RANDOM_STATE = 42

AVAILABLE_FEATURES = [
    "mfcc13",
    "mfcc20",
    "mfcc40",
    "chroma",
    "mel_spectrogram",
    "filterbank"
]

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# YMCM model definition
# ============================================================

def create_ymcm_model(input_shape, num_classes):
    """
    Final YMCM / custom-moeen model.

    This is the memory-safe version:
    - no additional layers
    - no L2 regularization to avoid memory issues
    - uses he_normal initialization
    - compatible with 224x224x3 feature images
    """

    model = Sequential([
        Conv2D(
            64, (7, 7),
            strides=(2, 2),
            activation="relu",
            padding="same",
            kernel_initializer="he_normal",
            input_shape=input_shape
        ),
        BatchNormalization(),
        MaxPooling2D((3, 3), strides=(2, 2)),
        Dropout(0.20),

        Conv2D(
            192, (5, 5),
            activation="relu",
            padding="same",
            kernel_initializer="he_normal"
        ),
        BatchNormalization(),
        MaxPooling2D((3, 3), strides=(2, 2)),
        Dropout(0.20),

        Conv2D(
            384, (3, 3),
            activation="relu",
            padding="same",
            kernel_initializer="he_normal"
        ),
        BatchNormalization(),

        Conv2D(
            256, (3, 3),
            activation="relu",
            padding="same",
            kernel_initializer="he_normal"
        ),
        BatchNormalization(),

        Conv2D(
            256, (3, 3),
            activation="relu",
            padding="same",
            kernel_initializer="he_normal"
        ),
        BatchNormalization(),
        MaxPooling2D((3, 3), strides=(2, 2)),
        Dropout(0.25),

        Flatten(),

        Dense(
            512,
            activation="relu",
            kernel_initializer="he_normal"
        ),
        Dropout(0.40),

        Dense(
            256,
            activation="relu",
            kernel_initializer="he_normal"
        ),
        Dropout(0.35),

        Dense(num_classes, activation="softmax")
    ], name="YMCM")

    return model


MODELS = {
    "YMCM": create_ymcm_model
}


# ============================================================
# Memory-safe NumPy Sequence
# ============================================================

class NumpyArraySequence(Sequence):
    """
    Memory-safe batch generator for .npy arrays.
    Works with standard np.ndarray and mmap_mode='r' arrays.
    """

    def __init__(self, X, y, num_classes, batch_size=4, shuffle=True):
        self.X = X
        self.y = y
        self.num_classes = num_classes
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.X))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        X_batch = np.asarray(self.X[batch_indices], dtype=np.float32)
        y_batch = np.asarray(self.y[batch_indices], dtype=np.int64)
        y_batch = to_categorical(y_batch, num_classes=self.num_classes)

        return X_batch, y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# ============================================================
# Data loading
# ============================================================

def get_fold_paths(cv_data_dir, fold_id, feature_name):
    fold_dir = os.path.join(cv_data_dir, f"fold_{fold_id}")
    train_dir = os.path.join(fold_dir, "train", feature_name)
    test_dir = os.path.join(fold_dir, "test", feature_name)

    paths = {
        "X_train": os.path.join(train_dir, f"{feature_name}_X_train.npy"),
        "y_train": os.path.join(train_dir, f"{feature_name}_y_train.npy"),
        "X_test": os.path.join(test_dir, f"{feature_name}_X_test.npy"),
        "y_test": os.path.join(test_dir, f"{feature_name}_y_test.npy"),
    }

    for key, path in paths.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing {key}: {path}")

    return paths


def load_fold_data(cv_data_dir, fold_id, feature_name, mmap=True):
    paths = get_fold_paths(cv_data_dir, fold_id, feature_name)

    mmap_mode = "r" if mmap else None

    X_train = np.load(paths["X_train"], mmap_mode=mmap_mode)
    y_train = np.load(paths["y_train"], mmap_mode=mmap_mode)
    X_test = np.load(paths["X_test"], mmap_mode=mmap_mode)
    y_test = np.load(paths["y_test"], mmap_mode=mmap_mode)

    return X_train, y_train, X_test, y_test


# ============================================================
# Training and evaluation per fold
# ============================================================

def train_one_fold(
    model_name,
    model_func,
    feature_name,
    fold_id,
    config
):
    print("\n" + "=" * 80)
    print(f"Training {model_name} | Feature: {feature_name} | Fold {fold_id}/{N_SPLITS}")
    print("=" * 80)

    tf.keras.backend.clear_session()
    gc.collect()

    X_train, y_train, X_test, y_test = load_fold_data(
        config["cv_data_dir"],
        fold_id,
        feature_name,
        mmap=True
    )

    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)
    print("X_test :", X_test.shape)
    print("y_test :", y_test.shape)

    batch_size = config["batch_size"]

    train_gen = NumpyArraySequence(
        X_train,
        y_train,
        num_classes=NUM_CLASSES,
        batch_size=batch_size,
        shuffle=True
    )

    test_gen = NumpyArraySequence(
        X_test,
        y_test,
        num_classes=NUM_CLASSES,
        batch_size=batch_size,
        shuffle=False
    )

    model = model_func(INPUT_SHAPE, NUM_CLASSES)

    learning_rate = config["learning_rate"]

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    fold_output_dir = os.path.join(config["output_dir"], model_name, feature_name, f"fold_{fold_id}")
    os.makedirs(fold_output_dir, exist_ok=True)

    checkpoint_path = os.path.join(fold_output_dir, f"best_{model_name}_{feature_name}_fold_{fold_id}.h5")

    callbacks = [
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_accuracy",
            save_best_only=True,
            mode="max",
            verbose=1
        ),
        EarlyStopping(
            monitor="val_accuracy",
            patience=config["patience"],
            restore_best_weights=True,
            mode="max",
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-7,
            verbose=1
        )
    ]

    start_time = time.time()

    history = model.fit(
        train_gen,
        validation_data=test_gen,
        epochs=config["epochs"],
        callbacks=callbacks,
        verbose=1
    )

    training_time = time.time() - start_time

    if os.path.exists(checkpoint_path):
        best_model = tf.keras.models.load_model(checkpoint_path)
    else:
        best_model = model

    test_loss, test_accuracy = best_model.evaluate(test_gen, verbose=0)

    y_prob = best_model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.asarray(y_test, dtype=np.int64)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    # Save history
    history_path = os.path.join(fold_output_dir, "training_history.csv")
    pd.DataFrame(history.history).to_csv(history_path, index=False)

    # Save classification report
    report = classification_report(
        y_true,
        y_pred,
        target_names=GENRES,
        output_dict=True,
        zero_division=0
    )
    report_path = os.path.join(fold_output_dir, "classification_report.csv")
    pd.DataFrame(report).transpose().to_csv(report_path)

    # Save confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_path = os.path.join(fold_output_dir, "confusion_matrix.csv")
    pd.DataFrame(cm, index=GENRES, columns=GENRES).to_csv(cm_path)

    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=GENRES, yticklabels=GENRES)
    plt.title(f"{model_name} - {feature_name} - Fold {fold_id}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(fold_output_dir, "confusion_matrix.png"), dpi=300)
    plt.close()

    # Save training curve
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history.get("accuracy", []), label="Train Accuracy")
    plt.plot(history.history.get("val_accuracy", []), label="Validation Accuracy")
    plt.title(f"Accuracy | Fold {fold_id}")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(history.history.get("loss", []), label="Train Loss")
    plt.plot(history.history.get("val_loss", []), label="Validation Loss")
    plt.title(f"Loss | Fold {fold_id}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(fold_output_dir, "training_curves.png"), dpi=300)
    plt.close()

    fold_result = {
        "model": model_name,
        "feature": feature_name,
        "fold": fold_id,
        "loss": float(test_loss),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "training_time_seconds": float(training_time),
        "best_val_accuracy": float(np.max(history.history.get("val_accuracy", [np.nan]))),
        "best_val_loss": float(np.min(history.history.get("val_loss", [np.nan]))),
        "checkpoint_path": checkpoint_path
    }

    with open(os.path.join(fold_output_dir, "fold_result.json"), "w") as f:
        json.dump(fold_result, f, indent=4)

    print("\nFold result:")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    # Memory cleanup
    del model, best_model, history, train_gen, test_gen
    del X_train, y_train, X_test, y_test
    tf.keras.backend.clear_session()
    gc.collect()

    return fold_result


# ============================================================
# Cross-validation summary
# ============================================================

def compute_ci95(values):
    """
    95% confidence interval using t-value for n=5 folds: 2.776.
    If a different n is used, this approximation remains acceptable for reporting,
    but scipy.stats.t can be used for an exact t critical value.
    """
    values = np.asarray(values, dtype=np.float64)
    n = len(values)

    mean = np.mean(values)
    std = np.std(values, ddof=1) if n > 1 else 0.0

    # t critical values for common small n; default normal approx if not listed.
    t_crit_lookup = {
        2: 12.706,
        3: 4.303,
        4: 3.182,
        5: 2.776,
        6: 2.571,
        7: 2.447,
        8: 2.365,
        9: 2.306,
        10: 2.262
    }

    t_crit = t_crit_lookup.get(n, 1.96)
    margin = t_crit * std / np.sqrt(n) if n > 1 else 0.0

    return mean, std, mean - margin, mean + margin


def summarize_cv_results(fold_results, output_dir):
    detailed_df = pd.DataFrame(fold_results)
    detailed_path = os.path.join(output_dir, "cv_detailed_fold_results.csv")
    detailed_df.to_csv(detailed_path, index=False)

    summary_rows = []

    for (model, feature), group in detailed_df.groupby(["model", "feature"]):
        row = {
            "Model": model,
            "Feature": feature,
            "Num_Folds": len(group)
        }

        for metric in ["accuracy", "precision", "recall", "f1", "loss"]:
            mean, std, ci_low, ci_high = compute_ci95(group[metric].values)
            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = std
            row[f"{metric}_variance"] = std ** 2
            row[f"{metric}_ci95_low"] = ci_low
            row[f"{metric}_ci95_high"] = ci_high
            row[f"{metric}_mean_percent"] = mean * 100 if metric != "loss" else mean
            row[f"{metric}_std_percent"] = std * 100 if metric != "loss" else std

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(output_dir, "cv_summary_mean_std_variance_ci95.csv")
    summary_df.to_csv(summary_path, index=False)

    # Plot accuracy mean ± std
    if not summary_df.empty:
        plt.figure(figsize=(10, 6))
        labels = summary_df["Model"] + " | " + summary_df["Feature"]
        means = summary_df["accuracy_mean"]
        stds = summary_df["accuracy_std"]

        plt.bar(labels, means, yerr=stds, capsize=5)
        plt.ylabel("Accuracy")
        plt.title("Cross-Validation Accuracy Mean ± Std")
        plt.xticks(rotation=45, ha="right")
        plt.ylim(0, 1)
        plt.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, "cv_accuracy_mean_std.png"), dpi=300)
        plt.close()

    print("\n" + "=" * 80)
    print("Cross-validation summary")
    print("=" * 80)
    print(summary_df)

    print("\nSaved:")
    print("Detailed fold results:", detailed_path)
    print("Summary:", summary_path)

    return detailed_df, summary_df


# ============================================================
# Main CV runner
# ============================================================

def run_cv_training(config):
    os.makedirs(config["output_dir"], exist_ok=True)

    all_fold_results = []

    for model_name in config["models"]:
        model_func = MODELS[model_name]

        for feature_name in config["features"]:
            for fold_id in range(1, config["n_splits"] + 1):
                try:
                    result = train_one_fold(
                        model_name=model_name,
                        model_func=model_func,
                        feature_name=feature_name,
                        fold_id=fold_id,
                        config=config
                    )
                    all_fold_results.append(result)

                except Exception as e:
                    print(f"Error: {model_name} | {feature_name} | Fold {fold_id}: {e}")
                    all_fold_results.append({
                        "model": model_name,
                        "feature": feature_name,
                        "fold": fold_id,
                        "error": str(e)
                    })

    valid_results = [r for r in all_fold_results if "accuracy" in r]

    if len(valid_results) == 0:
        raise RuntimeError("No successful fold results were generated.")

    detailed_df, summary_df = summarize_cv_results(valid_results, config["output_dir"])

    return {
        "fold_results": all_fold_results,
        "detailed_df": detailed_df,
        "summary_df": summary_df
    }


# ============================================================
# User configuration
# ============================================================

def get_user_configuration():
    print("\n" + "=" * 80)
    print("YMCM Cross-Validation Training Configuration")
    print("=" * 80)

    config = {}

    print("\nAvailable Models:")
    for i, model_name in enumerate(MODELS.keys(), 1):
        print(f"  {i}. {model_name}")

    model_choice = input("\nSelect models (comma-separated numbers or 'all', default=1): ").strip()
    if model_choice.lower() == "all":
        config["models"] = list(MODELS.keys())
    elif model_choice == "":
        config["models"] = ["YMCM"]
    else:
        indices = [int(x.strip()) - 1 for x in model_choice.split(",")]
        config["models"] = [list(MODELS.keys())[i] for i in indices if 0 <= i < len(MODELS)]

    print("\nAvailable Features:")
    for i, feature in enumerate(AVAILABLE_FEATURES, 1):
        print(f"  {i}. {feature}")

    feature_choice = input("\nSelect features (comma-separated numbers or 'all', default=5): ").strip()
    if feature_choice.lower() == "all":
        config["features"] = AVAILABLE_FEATURES
    elif feature_choice == "":
        config["features"] = ["mel_spectrogram"]
    else:
        indices = [int(x.strip()) - 1 for x in feature_choice.split(",")]
        config["features"] = [AVAILABLE_FEATURES[i] for i in indices if 0 <= i < len(AVAILABLE_FEATURES)]

    config["epochs"] = int(input("\nNumber of epochs (default=50): ") or 50)
    config["batch_size"] = int(input("Batch size (default=4): ") or 4)
    config["learning_rate"] = float(input("Learning rate (default=5e-5): ") or 5e-5)
    config["patience"] = int(input("Early stopping patience (default=18): ") or 18)

    config["n_splits"] = int(input("Number of CV folds (default=5): ") or 5)
    config["cv_data_dir"] = input(f"CV data directory (default={CV_DATA_DIR}): ") or CV_DATA_DIR
    config["output_dir"] = input(f"Output directory (default={OUTPUT_DIR}): ") or OUTPUT_DIR

    os.makedirs(config["output_dir"], exist_ok=True)

    print("\nConfiguration Summary:")
    print(json.dumps(config, indent=4))

    confirm = input("\nConfirm configuration? (yes/no, default=yes): ").strip().lower() or "yes"
    if confirm != "yes":
        return None

    with open(os.path.join(config["output_dir"], "cv_training_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    return config


# ============================================================
# Execute
# ============================================================

def main():
    configure_gpu()

    config = get_user_configuration()
    if config is None:
        print("Cancelled.")
        return

    outputs = run_cv_training(config)

    print("\nTraining completed.")
    print("Output directory:", config["output_dir"])

    # Print reviewer-ready summary
    summary_df = outputs["summary_df"]
    print("\nReviewer-ready mean ± std:")
    for _, row in summary_df.iterrows():
        print(
            f"{row['Model']} | {row['Feature']}: "
            f"Accuracy = {row['accuracy_mean_percent']:.2f} ± {row['accuracy_std_percent']:.2f}% | "
            f"F1 = {row['f1_mean_percent']:.2f} ± {row['f1_std_percent']:.2f}%"
        )

if __name__ == "__main__":
    main()


GPU memory growth enabled for 1 GPU(s).
TensorFlow version: 2.9.0

YMCM Cross-Validation Training Configuration

Available Models:
  1. YMCM



Select models (comma-separated numbers or 'all', default=1):  1



Available Features:
  1. mfcc13
  2. mfcc20
  3. mfcc40
  4. chroma
  5. mel_spectrogram
  6. filterbank



Select features (comma-separated numbers or 'all', default=5):  1

Number of epochs (default=50):  
Batch size (default=4):  
Learning rate (default=5e-5):  
Early stopping patience (default=18):  
Number of CV folds (default=5):  
CV data directory (default=/root/autodl-tmp/cv_split_data/):  
Output directory (default=/root/autodl-tmp/YMCM_CV_results/):  



Configuration Summary:
{
    "models": [
        "YMCM"
    ],
    "features": [
        "mfcc13"
    ],
    "epochs": 50,
    "batch_size": 4,
    "learning_rate": 5e-05,
    "patience": 18,
    "n_splits": 5,
    "cv_data_dir": "/root/autodl-tmp/cv_split_data/",
    "output_dir": "/root/autodl-tmp/YMCM_CV_results/"
}



Confirm configuration? (yes/no, default=yes):  yes



Training YMCM | Feature: mfcc13 | Fold 1/5
X_train: (4600, 224, 224, 3)
y_train: (4600,)
X_test : (1150, 224, 224, 3)
y_test : (1150,)
Epoch 1/50
1150/1150 [==============================] - ETA: 0s - loss: 2.0897 - accuracy: 0.3578
Epoch 1: val_accuracy improved from -inf to 0.34522, saving model to /root/autodl-tmp/YMCM_CV_results/YMCM/mfcc13/fold_1/best_YMCM_mfcc13_fold_1.h5
1150/1150 [==============================] - 13s 9ms/step - loss: 2.0897 - accuracy: 0.3578 - val_loss: 1.6577 - val_accuracy: 0.3452 - lr: 5.0000e-05
Epoch 2/50
1146/1150 [============================>.] - ETA: 0s - loss: 1.3112 - accuracy: 0.4544
Epoch 2: val_accuracy improved from 0.34522 to 0.47565, saving model to /root/autodl-tmp/YMCM_CV_results/YMCM/mfcc13/fold_1/best_YMCM_mfcc13_fold_1.h5
1150/1150 [==============================] - 10s 9ms/step - loss: 1.3121 - accuracy: 0.4541 - val_loss: 1.4701 - val_accuracy: 0.4757 - lr: 5.0000e-05
Epoch 3/50
1149/1150 [============================>.] - ETA: 0s - l